# Project A — WestQuant Representation Scheduler (QoolQit)

**QoolQit version: 1.4.0**

For a fixed logical Hamiltonian, search over physical embeddings and
hardware realizations to find QoolQit programs that outperform a single
default embedding across multiple objectives (interaction fidelity,
logical fidelity, compilation, duration, solution quality, robustness).

Uses successive halving to avoid expensive emulation of all candidates.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, qoolqit
print('QoolQit version:', qoolqit.__version__)
from westquant_qoolqit.common import BinaryQuadraticHamiltonian, solve_exact, QOOLQIT_VERSION
from westquant_qoolqit.representation_scheduler import RepresentationScheduler, HalvingConfig
from westquant_qoolqit.benchmarks import mwis_path
from qoolqit import AnalogDeviceWithDMM
assert QOOLQIT_VERSION == qoolqit.__version__

## 1. Problem setup

In [ ]:
h, info = mwis_path(n=5, seed=42)
exact = solve_exact(h)
print('Optimum:', exact.optimum_states.tolist(), 'E=', exact.optimum_energy)

## 2. Representation search

Generate embedding candidates from QoolQit's `InteractionEmbedder`,
`SpringLayoutEmbedder`, and `Blade`, then score via successive halving:
- Stage 0: cheap geometric + interaction metrics
- Stage 1: logical fidelity
- Stage 2: compilation
- Stage 3: local emulation
- Stage 4: robustness under coordinate perturbation

In [ ]:
scheduler = RepresentationScheduler(
    embedders=['interaction', 'spring', 'blade'], budget=12, seed=42, num_shots=200,
    halving=HalvingConfig(stage0_keep=1.0, stage2_keep=6, stage3_keep=3, stage4_keep=2),
)
mwis_weights = np.asarray(info['weights'])
result = scheduler.search(h, device=AnalogDeviceWithDMM(),
                         run_emulation=True, run_robustness=True, robustness_samples=3,
                         mwis_weights=mwis_weights)
print('Results:')
for s in result.scores:
    if s.compilation_success and s.ground_state_probability is not None:
        print(f'  {s.candidate_id:20s} frob={s.frobenius_error:.4f} '
              f'p_opt={s.ground_state_probability:.4f} robust={s.robust_interaction_rank_mean:.3f}')
best = result.best('ground_state_probability')
if best: print(f'Best: {best.candidate_id}')

## 3. Pareto analysis

In [ ]:
front, idx = result.pareto_front()
print('Pareto front size:', len(front))
print('Labels:', result.labels())

## 4. Conclusion

Different embeddings of the same Hamiltonian yield materially different
solution probabilities and robustness. Treating the embedding as an
optimization variable (rather than a fixed choice) improves results.